# LSTM Model Training: AADT Forecasting

This notebook walks through the full training pipeline interactively, with inline visualizations at each step.

**Pipeline steps:**
1. Load and scale data
2. Build sequences for LSTM input
3. Define and compile the stacked LSTM model
4. Train with early stopping
5. Evaluate on held-out test set
6. Visualize predictions and residuals
7. Forecast next year

> For production use, run `python code/train.py` directly with CLI arguments.

In [ ]:
import sys, os, math
sys.path.append(os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from tensorflow.keras.callbacks import EarlyStopping

from utils.data_loader import load_adt_csv
from utils.metrics import regression_metrics
from utils.plot_utils import plot_union, plot_test_vs_pred, plot_loss

%matplotlib inline
plt.rcParams['figure.dpi'] = 120
print('All imports successful')

## 1. Configuration

All key parameters are set here. Adjust `SEQUENCE_LENGTH` and `LSTM_UNITS` based on EDA findings.

In [ ]:
# --- Config ---
CSV_PATH        = '../input_data/raw/sample_aadt.csv'
SEQUENCE_LENGTH = 12      # Number of past years used as input
TRAIN_SPLIT     = 0.70    # 70% train, 30% test
LSTM_UNITS      = 64      # Hidden units per LSTM layer
DENSE_UNITS     = 25      # Units in pre-output dense layer
BATCH_SIZE      = 8
MAX_EPOCHS      = 300
PATIENCE        = 25      # Early stopping patience
SAVE_MODEL      = True
MODEL_PATH      = '../output/models/latest_lstm.keras'

print(f'Sequence length : {SEQUENCE_LENGTH} years')
print(f'Train split     : {TRAIN_SPLIT:.0%}')
print(f'LSTM units      : {LSTM_UNITS}')

## 2. Load and Scale Data

In [ ]:
df = load_adt_csv(CSV_PATH)
print(f'Loaded {len(df)} observations | Years: {df.index.min()}–{df.index.max()}')
print(f'AADT range: {df.iloc[:,0].min():,.0f} – {df.iloc[:,0].max():,.0f}')

values = df.values
scaler = MinMaxScaler((0, 1))
values_scaled = scaler.fit_transform(values)

train_len = math.ceil(len(values_scaled) * TRAIN_SPLIT)
print(f'\nTrain samples : {train_len} | Test samples: {len(values_scaled) - train_len}')

## 3. Build Sequences

The LSTM takes windows of `SEQUENCE_LENGTH` consecutive years as input to predict the next year's AADT.

In [ ]:
def to_sequences(data_2d, seq_len, pred_index=0):
    X, y = [], []
    for i in range(seq_len, data_2d.shape[0]):
        X.append(data_2d[i - seq_len:i, :])
        y.append(data_2d[i, pred_index])
    return np.array(X), np.array(y)

train_data = values_scaled[:train_len, :]
test_data  = values_scaled[train_len - SEQUENCE_LENGTH:, :]

X_train, y_train = to_sequences(train_data, SEQUENCE_LENGTH)
X_test,  y_test  = to_sequences(test_data,  SEQUENCE_LENGTH)

print(f'X_train shape : {X_train.shape}  (samples × timesteps × features)')
print(f'X_test shape  : {X_test.shape}')

## 4. Build and Compile Model

In [ ]:
model = Sequential([
    LSTM(LSTM_UNITS, return_sequences=True,  input_shape=(X_train.shape[1], X_train.shape[2])),
    LSTM(LSTM_UNITS, return_sequences=True),
    LSTM(LSTM_UNITS, return_sequences=False),
    Dense(DENSE_UNITS, activation='relu'),
    Dense(1)
])

model.compile(optimizer='adam', loss='mean_squared_error')
model.summary()

## 5. Train with Early Stopping

In [ ]:
early_stop = EarlyStopping(
    monitor='val_loss', mode='min',
    patience=PATIENCE, verbose=1, restore_best_weights=True
)

history = model.fit(
    X_train, y_train,
    batch_size=BATCH_SIZE,
    epochs=MAX_EPOCHS,
    validation_split=0.2,
    shuffle=True,
    verbose=0,
    callbacks=[early_stop]
)

stopped_epoch = len(history.history['val_loss'])
print(f'Training stopped at epoch {stopped_epoch}')
print(f'Best val_loss: {min(history.history["val_loss"]):.6f}')

## 6. Training History

In [ ]:
plot_loss(history)

## 7. Evaluate on Test Set

In [ ]:
y_pred_scaled = model.predict(X_test)
y_pred = scaler.inverse_transform(y_pred_scaled)
y_true = scaler.inverse_transform(y_test.reshape(-1, 1))

metrics = regression_metrics(y_true, y_pred)
print('Test Set Metrics:')
print('-' * 30)
for k, v in metrics.items():
    print(f'  {k:<8}: {v:>10.2f}')

## 8. Prediction Plots

In [ ]:
# Build union dataframe for full-series plot
train = df.iloc[:train_len + 1].rename(columns={df.columns[0]: 'x_train'})
valid = df.iloc[train_len:].rename(columns={df.columns[0]: 'y_test'}).copy()
valid.insert(1, 'y_pred', y_pred.flatten())
valid.insert(1, 'residuals', valid['y_pred'].values - y_true.flatten())
union = pd.concat([train, valid])

plot_union(union, ylabel='AADT (vehicles/day)')
plot_test_vs_pred(valid, ylabel='AADT (vehicles/day)')

## 9. Residual Analysis

In [ ]:
residuals = y_true.flatten() - y_pred.flatten()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].bar(df.index[train_len:train_len + len(residuals)], residuals, color='steelblue', alpha=0.8)
axes[0].axhline(0, color='black', linewidth=0.8)
axes[0].set_title('Residuals Over Time')
axes[0].set_xlabel('Year')
axes[0].set_ylabel('Residual (Actual - Predicted)')
axes[0].grid(axis='y', alpha=0.4)

axes[1].hist(residuals, bins=8, color='steelblue', edgecolor='white', alpha=0.8)
axes[1].axvline(0, color='tomato', linewidth=1.2, linestyle='--')
axes[1].set_title('Residual Distribution')
axes[1].set_xlabel('Residual')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

print(f'Mean residual : {residuals.mean():+.1f} (bias check — should be near 0)')
print(f'Std residual  : {residuals.std():.1f}')

## 10. Next-Year Forecast

In [ ]:
last_sequence = values_scaled[-SEQUENCE_LENGTH:].reshape(1, SEQUENCE_LENGTH, 1)
next_scaled   = model.predict(last_sequence)
next_aadt     = scaler.inverse_transform(next_scaled).flatten()[0]
next_year     = int(df.index.max()) + 1

print(f'Forecast for {next_year}: {next_aadt:,.0f} vehicles/day')

## 11. Save Model

In [ ]:
if SAVE_MODEL:
    os.makedirs(os.path.dirname(MODEL_PATH), exist_ok=True)
    model.save(MODEL_PATH)
    print(f'Model saved to: {MODEL_PATH}')